In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.applications import MobileNetV2
from keras import layers,models,optimizers
from keras.utils import image_dataset_from_directory

In [2]:
os.environ['KAGGLE_USERNAME'] = "https://www.kaggle.com/shubhanshishaurya"
os.environ['KAGGLE_KEY'] =""

In [3]:
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset

Dataset URL: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset
License(s): copyright-authors
new-plant-diseases-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [4]:
!unzip -q *.zip -d ./dataset

replace ./dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train/Apple___Apple_scab/00075aa8-d81a-4184-8541-b692b78d398a___FREC_Scab 3335.JPG? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [5]:
import os
print(os.listdir("./dataset"))

['new plant diseases dataset(augmented)', 'New Plant Diseases Dataset(Augmented)', 'test']


In [9]:
DATA_DIR = "/content/dataset/new plant diseases dataset(augmented)/New Plant Diseases Dataset(Augmented)/train"
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print(f"\nTarget Classes Found ({NUM_CLASSES}): {class_names}")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

Found 70295 files belonging to 38 classes.
Using 56236 files for training.
Found 70295 files belonging to 38 classes.
Using 14059 files for validation.

Target Classes Found (38): ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacteri

In [10]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

In [11]:
model = models.Sequential([
    layers.Rescaling(1./255),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

In [12]:
print("\n--- Training Top Classification Layers ---")
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(train_ds, epochs=5, validation_data=val_ds)

print("\n--- Fine-Tuning Top Feature Extraction Blocks ---")
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(train_ds, epochs=3, validation_data=val_ds)


--- Training Top Classification Layers ---
Epoch 1/5
3515/3515 ━━━━━━━━━━━━━━━━━━━━ 146s 36ms/step - accuracy: 0.7465 - loss: 0.8269 - val_accuracy: 0.9220 - val_loss: 0.2424
Epoch 2/5
3515/3515 ━━━━━━━━━━━━━━━━━━━━ 86s 25ms/step - accuracy: 0.8530 - loss: 0.4480 - val_accuracy: 0.9323 - val_loss: 0.1998
Epoch 3/5
3515/3515 ━━━━━━━━━━━━━━━━━━━━ 90s 26ms/step - accuracy: 0.8708 - loss: 0.3914 - val_accuracy: 0.9372 - val_loss: 0.1883
Epoch 4/5
3515/3515 ━━━━━━━━━━━━━━━━━━━━ 143s 26ms/step - accuracy: 0.8820 - loss: 0.3558 - val_accuracy: 0.9433 - val_loss: 0.1662
Epoch 5/5
3515/3515 ━━━━━━━━━━━━━━━━━━━━ 139s 25ms/step - accuracy: 0.8870 - loss: 0.3350 - val_accuracy: 0.9437 - val_loss: 0.1696

--- Fine-Tuning Top Feature Extraction Blocks ---
Epoch 1/3
3515/3515 ━━━━━━━━━━━━━━━━━━━━ 157s 38ms/step - accuracy: 0.7664 - loss: 0.8982 - val_accuracy: 0.9383 - val_loss: 0.1843
Epoch 2/3
3515/3515 ━━━━━━━━━━━━━━━━━━━━ 174s 34ms/step - accuracy: 0.8740 - loss: 0.3971 - val_accuracy: 0.9577 - 

In [14]:
MODEL_OUTPUT_PATH = "disease_classifier.h5"
model.save(MODEL_OUTPUT_PATH)
print(f"\n✅ SUCCESS: Model saved successfully to current runtime directory as '{MODEL_OUTPUT_PATH}'")


✅ SUCCESS: Model saved successfully to current runtime directory as 'disease_classifier.h5'


In [15]:
from google.colab import files
files.download('disease_classifier.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
import os

COLAB_DATA_DIR = "/content/dataset/new plant diseases dataset(augmented)/New Plant Diseases Dataset(Augmented)/train"

if os.path.exists(COLAB_DATA_DIR):
    class_names = sorted([f for f in os.listdir(COLAB_DATA_DIR) if os.path.isdir(os.path.join(COLAB_DATA_DIR, f))])

    with open("classes.txt", "w") as f:
        for name in class_names:
            f.write(f"{name}\n")

    print(f"✅ Created classes.txt with {len(class_names)} classes successfully!")
else:
    print("❌ Could not find the dataset path. Make sure COLAB_DATA_DIR matches your training path.")

✅ Created classes.txt with 38 classes successfully!


In [17]:
from google.colab import files
files.download('classes.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>